In [ ]:
# ============================================================
#  CROSS-DATABASE DEDUPLICATION v2 — full synthesis  (Google Colab)
#
#  Combines the best of three designs:
#    - DOI-first staging + cluster-level DOI-conflict guard + union-find   (rigorous transitivity)
#    - risky/generic-title blocklist + cached normalization                (from the 2-pass engine)
#    - escalating author fingerprints as a SAFETY GATE for title/fuzzy     (precision)
#    - per-stage metrics                                                   (auditable, PRISMA-ready)
#
#  Matching ladder (each gated to avoid false merges):
#    Stage 1  exact DOI                              (no author gate; shared DOI is proof)
#    Stage 2  title_clean + authors_fp + year        (exact, author-gated)
#    Stage 3  title_clean + year, >=1 shared author  (catches author-format differences)
#    Stage 4  fuzzy title (token_sort_ratio>=90),    (year +/-1, REQUIRES shared author,
#             cluster-level DOI guard, blocked        never crosses a DOI boundary)
#
#  Input : master_search_results_with_ids.xlsx
#  Output: master_dedupe_audit.xlsx, master_canonical.xlsx, qc/*.xlsx
# ============================================================

## ---- 1. setup ----
from google.colab import drive
drive.mount('/content/drive')
!pip -q install rapidfuzz openpyxl pandas

import os, re, time, unicodedata
import pandas as pd, numpy as np
from rapidfuzz import fuzz
pd.set_option('display.max_columns', None)

## ---- 2. config ----
FOLDER = "/content/drive/MyDrive/COGNOSX/ticks"  # <-- EDIT
IN_PATH  = os.path.join(FOLDER, "data_raw/combined.parquet")
OUT_DIR  = os.path.join(FOLDER, "preprocessing_data")
QC_DIR   = os.path.join(OUT_DIR, "qc")
os.makedirs(QC_DIR, exist_ok=True)
AUDIT_PATH = os.path.join(OUT_DIR, "master_dedupe_audit.xlsx")
CANON_PATH = os.path.join(OUT_DIR, "master_canonical.xlsx")

# tunable parameters
FUZZY_THRESHOLD       = 90    # token_sort_ratio 0-100 for Stage 4
FUZZY_YEAR_TOLERANCE  = 3     # +/- years allowed for fuzzy matches
MIN_TITLE_TOKENS      = 3     # risky-title gate
MIN_TITLE_CHARS       = 15    # risky-title gate
AUTHOR_FP_MODE        = "last_initial"   # tokenbag | last_initial | last

## ---- 3. load ----
df = pd.read_parquet(IN_PATH).fillna("").reset_index(drop=True)
print("Loaded:", len(df), "rows,", len(df.columns), "columns")
if "record_id" not in df.columns:
    df["record_id"] = [f"REC-{i+1:06d}" for i in range(len(df))]

## ---- 4. normalization helpers (cached once) ----
_WS = re.compile(r"\s+"); _PUNCT = re.compile(r"[^\w\s]", re.UNICODE)
NA_LIKE = {"", "none", "null", "nan", "n/a", "[]", "{}", "na"}

def strip_acc(x):
    return "".join(c for c in unicodedata.normalize("NFKD", x) if not unicodedata.combining(c))

def clean_title(s):
    s = strip_acc(str(s or "").lower().strip())
    if s in NA_LIKE: return ""
    s = _PUNCT.sub(" ", s); s = _WS.sub(" ", s).strip()
    return s

def norm_doi(d):
    d = str(d or "").strip().lower()
    for p in ("https://doi.org/","http://dx.doi.org/","https://dx.doi.org/","doi:"):
        d = d.replace(p, "")
    d = d.strip()
    return d if d.startswith("10.") else ""

def norm_year(y):
    m = re.search(r"(19|20)\d{2}", str(y or "")); return m.group(0) if m else ""

# --- journal normalizer: lowercase, strip accents/punctuation/parentheticals,
#     drop noise words so abbreviated and full forms move closer. Used only as a
#     CONFIRMING signal (never to block a merge), because coverage is partial and
#     abbreviations vary across databases. ---
JOURNAL_STOP = {"the","of","and","for","in","on","a","journal","review","reviews","annual",
    "international","official","series","section","part","online","print","electronic","edition",
    "proceedings","bulletin","archives","annals","acta","revista","rev","jnl","j"}
JOURNAL_NA = {"", "none", "null", "nan", "n/a", "na", "med"}
def clean_journal(s):
    s = strip_acc(str(s or "").lower().strip())
    if s in JOURNAL_NA: return ""
    s = re.sub(r"\(.*\)", " ", s)        # drop "(Online)", "(Print)" etc.
    s = _PUNCT.sub(" ", s); s = _WS.sub(" ", s).strip()
    return " ".join(t for t in s.split() if t and t not in JOURNAL_STOP)

def journal_agree(a, b):
    """True if journals clearly agree (fuzzy), else False. Missing -> False (never blocks)."""
    if not a or not b: return False
    if a == b: return True
    return fuzz.token_set_ratio(a, b) >= 85

# --- risky / generic title blocklist (extend freely) ---
GENERIC_TITLES = frozenset({
    "front matter","back matter","summary","summaries","reviews in brief","publications received",
    "editorial","introduction","preface","contents","table of contents","index","book review",
    "letter","news","announcement","abstract","poster","supplement","no title","retracted",
    "bibliographie","corrigendum","erratum","obituary","in this issue","highlights",
    "acknowledgements","author index","subject index","calendar","meeting","conference","abstracts",
})
def is_risky(t):
    if not t: return True
    if t in GENERIC_TITLES: return True
    if len(t.split()) < MIN_TITLE_TOKENS: return True
    if len(t) < MIN_TITLE_CHARS: return True
    return False

# --- author fingerprint (3 modes; last_initial is the safe default) ---
def author_fp(authors_flat, mode=AUTHOR_FP_MODE):
    a = str(authors_flat or "").strip()
    if a.lower() in NA_LIKE: return ""
    out = []
    for au in re.split(r"[;]", a):
        au = strip_acc(au.lower().strip())
        if not au or au in NA_LIKE: continue
        if "," in au:
            left, right = au.split(",", 1)
            left  = _WS.sub(" ", _PUNCT.sub(" ", left)).strip()
            right = _WS.sub(" ", _PUNCT.sub(" ", right)).strip()
            lt = [t for t in left.split() if t]; rt = [t for t in right.split() if t]
            if not lt: continue
            if mode == "last":
                out.append(lt[0])
            else:
                if not rt: continue
                if mode == "tokenbag":
                    out.append("_".join(sorted(lt + rt)))
                else:  # last_initial
                    out.append(f"{lt[0]}|{rt[0][:1]}")
        else:
            au = _WS.sub(" ", _PUNCT.sub(" ", au)).strip(); toks = [t for t in au.split() if t]
            if len(toks) < 2:
                if mode == "last" and toks: out.append(toks[-1])
                continue
            if mode == "last":
                out.append(toks[-1])
            elif mode == "tokenbag":
                out.append("_".join(sorted(toks)))
            else:  # last_initial
                out.append(f"{toks[-1]}|{toks[0][:1]}")
    return ";".join(sorted(set(out)))

def fp_overlap(a, b):
    if not a or not b: return 0
    return len(set(a.split(";")) & set(b.split(";")))

t0 = time.perf_counter()
df["title_clean"] = df["oa_title"].map(clean_title)
df.loc[df["title_clean"].map(is_risky), "title_clean"] = ""   # blank risky titles up front
df["doi_norm"]   = df["oa_doi"].map(norm_doi)
df["year_norm"]  = df["oa_year"].map(norm_year)
df["authors_fp"] = df["oa_authors"].map(author_fp)
df["journal_clean"] = df["oa_journal"].map(clean_journal)
df["title_first_token"] = df["title_clean"].map(lambda t: t.split(" ")[0] if t else "")
print(f"Normalization: {time.perf_counter()-t0:.1f}s | risky titles blanked: {(df['title_clean']=='').sum()}")

## ---- 5. union-find with cluster-level DOI tracking ----
parent = {}; cluster_dois = {}
def find(x):
    parent.setdefault(x, x); r = x
    while parent[r] != r: r = parent[r]
    while parent[x] != r: parent[x], x = r, parent[x]
    return r
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb
        cluster_dois.setdefault(rb, set()).update(cluster_dois.get(ra, set()))
def doi_conflict(a, b):
    da = cluster_dois.get(find(a), set()); db = cluster_dois.get(find(b), set())
    return bool(da and db and da != db and not (da & db))

for i in df.index:
    find(i)
    d = df.at[i, "doi_norm"]
    if d: cluster_dois[i] = {d}

M = {}

## Stage 1: exact DOI (no author gate)
n = 0
for doi, g in df[df["doi_norm"].ne("")].groupby("doi_norm"):
    ix = g.index.tolist()
    for j in ix[1:]: union(ix[0], j); n += 1
M["1_doi_exact"] = n

## Stage 2: title_clean + authors_fp + year (exact, author-gated)
n = 0
el = df["title_clean"].ne("") & df["authors_fp"].ne("") & df["year_norm"].ne("")
key = df["title_clean"] + "||" + df["authors_fp"] + "||" + df["year_norm"]
for k, g in df[el].groupby(key[el]):
    ix = g.index.tolist()
    for j in ix[1:]:
        if find(ix[0]) != find(j) and not doi_conflict(ix[0], j):
            union(ix[0], j); n += 1
M["2_title_fp_year"] = n

## Stage 2: title_clean + journal_clean + year
n = 0
el = df["title_clean"].ne("") & df["journal_clean"].ne("") & df["year_norm"].ne("")
key = df["title_clean"] + "||" + df["journal_clean"] + "||" + df["year_norm"]
for k, g in df[el].groupby(key[el]):
    ix = g.index.tolist()
    for j in ix[1:]:
        if find(ix[0]) != find(j) and not doi_conflict(ix[0], j):
            union(ix[0], j); n += 1
M["2_title_journal_year"] = n

## Stage 3: title_clean + year, gated by >=1 shared author (catches author-format diffs)
n = 0
el = df["title_clean"].ne("") & df["year_norm"].ne("")
key = df["title_clean"] + "||" + df["year_norm"]
for k, g in df[el].groupby(key[el]):
    ix = g.index.tolist()
    if len(ix) < 2: continue
    for a in range(len(ix)):
        for b in range(a + 1, len(ix)):
            ia, ib = ix[a], ix[b]
            if find(ia) == find(ib) or doi_conflict(ia, ib): continue
            fa, fb = df.at[ia, "authors_fp"], df.at[ib, "authors_fp"]
            if fa and fb and fp_overlap(fa, fb) < 1: continue   # gate when both have authors
            union(ia, ib); n += 1
M["3_title_year_authorgate"] = n

## Stage 4: fuzzy title, year+/-1, REQUIRES shared author, cluster-level DOI guard
n = 0
cand = df[df["title_clean"].ne("")]
for tok, block in cand.groupby("title_first_token"):
    if tok == "": continue
    ix = block.index.tolist()
    if len(ix) < 2: continue
    info = [(i, df.at[i,"title_clean"], df.at[i,"year_norm"], df.at[i,"authors_fp"], df.at[i,"journal_clean"]) for i in ix]
    for a in range(len(info)):
        ia, ta, ya, fa, ja = info[a]
        for b in range(a + 1, len(info)):
            ib, tb, yb, fb, jb = info[b]
            if find(ia) == find(ib) or doi_conflict(ia, ib): continue
            if ya and yb and abs(int(ya) - int(yb)) > FUZZY_YEAR_TOLERANCE: continue
            # confirmation: EITHER a shared author OR clearly-agreeing journal
            shared_author = fa and fb and fp_overlap(fa, fb) >= 1
            same_journal  = journal_agree(ja, jb)
            if not (shared_author or same_journal): continue
            if abs(len(ta) - len(tb)) > 25: continue
            if fuzz.token_sort_ratio(ta, tb) >= FUZZY_THRESHOLD:
                union(ia, ib); n += 1
M["4_fuzzy_gated"] = n

## ---- 6. assign study_id, canonical, provenance ----
df["cluster"] = [find(i) for i in df.index]

# canonical-row score: DOI >> type priority + metadata completeness + abstract
TYPE_PRIORITY = {"journal article":10,"research article":10,"article":10,"review":8,
    "review article":8,"systematic review":8,"book chapter":5,"book":5,"conference paper":4,
    "proceedings":4,"preprint":3,"posted content":3,"report":2,"thesis":2,"dataset":1}
def type_pri(t):
    t = str(t or "").strip().lower(); hits = [v for k,v in TYPE_PRIORITY.items() if k in t]
    return max(hits) if hits else 0
CORE = ["title","authors","year","doi","journal","abstract","keywords","document_type"]
df["_meta"] = sum((df[c].astype(str).str.len() > 0).astype(int) for c in CORE if c in df.columns)
df["_score"] = (df["doi_norm"].ne("").astype(int)*100 + df["document_type"].map(type_pri)*3
                + df["_meta"]*5 + (df["abstract"].astype(str).str.len() > 0).astype(int)*6)
canon_idx = df.sort_values(["_score","record_id"], ascending=[False,True]).groupby("cluster").head(1).index
df["is_canonical"] = df.index.isin(canon_idx)

order = {c: n for n, c in enumerate(df["cluster"].tolist())}
uniq  = sorted(df["cluster"].unique(), key=lambda c: order[c])
cl2stu = {c: f"STU-{i+1:06d}" for i, c in enumerate(uniq)}
df["study_id"] = df["cluster"].map(cl2stu)
found = df.groupby("study_id")["source_database"].apply(lambda s: "; ".join(sorted(set(s))))
df["found_in"]  = df["study_id"].map(found)
df["n_sources"] = df.groupby("study_id")["source_database"].transform("nunique")
df["n_records"] = df.groupby("study_id")["record_id"].transform("count")

## ---- 7. report ----
nr, ns = len(df), df["study_id"].nunique()
print("\n=== STAGE MERGES ===")
for k, v in M.items(): print(f"  {k}: {v}")
print("\n=== PRISMA identification ===")
print(f"Records identified: {nr}")
print(df["source_database"].value_counts().to_string())
print(f"\nDuplicates removed:        {nr - ns}")
print(f"Records after de-dup:      {ns}")
print("\nStudies by #databases:")
print(df.drop_duplicates('study_id')['n_sources'].value_counts().sort_index().to_string())
print(f"\nLargest study: {df.groupby('study_id').size().max()} records")

## ---- 8. QC files ----
study_has_doi = df.groupby("study_id")["doi_norm"].transform(lambda s: s.ne("").any())
qc_canon_no_doi = df[df["is_canonical"] & df["doi_norm"].eq("") & study_has_doi]
tt = df.groupby("study_id")["title_clean"].nunique()
qc_multi_title  = df[df["study_id"].isin(tt[tt > 1].index)]
qc_big          = df[df["n_records"] >= 8]    # unusually large clusters to eyeball
print(f"\n=== QC ===")
print(f"Canonical missing DOI while dup has one: {len(qc_canon_no_doi)}")
print(f"Rows in multi-distinct-title studies:    {len(qc_multi_title)}")
print(f"Rows in large studies (>=8 records):     {len(qc_big)}")

## ---- 9. save ----
def safe_xlsx(frame, path, sheet):
    ILLEGAL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
    out = frame.copy()
    for c in out.columns:
        out[c] = out[c].map(lambda v: ILLEGAL.sub(" ", str(v))[:32000])
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        out.to_excel(xw, sheet_name=sheet, index=False)

front = ["study_id","record_id","is_canonical","n_sources","n_records","found_in",
         "source_database","source_record_id","title","authors","year","doi","journal",
         "abstract","keywords","url","oa_type","oa_funders","oa_cited_by_count"]
front = [c for c in front if c in df.columns]
helper = ["title_clean","doi_norm","year_norm","authors_fp","title_first_token",
          "cluster","_meta","_score"]
rest = [c for c in df.columns if c not in front + helper]
audit = df[front + rest]
safe_xlsx(audit, AUDIT_PATH, "dedupe_audit")
safe_xlsx(audit[df["is_canonical"]], CANON_PATH, "canonical")
safe_xlsx(qc_canon_no_doi[front], os.path.join(QC_DIR,"qc_canonical_missing_doi.xlsx"), "qc")
safe_xlsx(qc_multi_title[front],  os.path.join(QC_DIR,"qc_multi_title_studies.xlsx"), "qc")
safe_xlsx(qc_big[front],          os.path.join(QC_DIR,"qc_large_studies.xlsx"), "qc")

print("\nSaved:")
print(" ", AUDIT_PATH, f"({len(audit)} rows)")
print(" ", CANON_PATH, f"({df['is_canonical'].sum()} unique studies)")
print(" ", QC_DIR, "(3 review files)")

In [ ]:
## ---- 9. save ----
def safe_xlsx(frame, path, sheet):
    ILLEGAL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
    out = frame.copy()
    for c in out.columns:
        out[c] = out[c].map(lambda v: ILLEGAL.sub(" ", str(v))[:32000])
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        out.to_excel(xw, sheet_name=sheet, index=False)

front = ["study_id","record_id","is_canonical","n_sources","n_records","found_in",
         "source_database","source_record_id","title","authors","year","doi","journal",
         "abstract","keywords","url","oa_type","oa_funders","oa_cited_by_count"]
front = [c for c in front if c in df.columns]
helper = ["title_clean","doi_norm","year_norm","authors_fp","title_first_token",
          "cluster","_meta","_score"]
rest = [c for c in df.columns if c not in front + helper]
audit = df[front + rest]
safe_xlsx(audit, AUDIT_PATH, "dedupe_audit")
safe_xlsx(audit[df["is_canonical"]], CANON_PATH, "canonical")
safe_xlsx(qc_canon_no_doi[front], os.path.join(QC_DIR,"qc_canonical_missing_doi.xlsx"), "qc")
safe_xlsx(qc_multi_title[front],  os.path.join(QC_DIR,"qc_multi_title_studies.xlsx"), "qc")
safe_xlsx(qc_big[front],          os.path.join(QC_DIR,"qc_large_studies.xlsx"), "qc")

print("\nSaved:")
print(" ", AUDIT_PATH, f"({len(audit)} rows)")
print(" ", CANON_PATH, f"({df['is_canonical'].sum()} unique studies)")
print(" ", QC_DIR, "(3 review files)")

In [ ]:
df